# Fine-Tune Llama 3.1 8B on PubMedQA (QLoRA)

**MEGA-RAG Project** — Fine-tune Llama 3.1 8B Instruct for medical yes/no/maybe classification using QLoRA on Kaggle's free T4 GPU.

## Setup Steps (Before Running)
1. **Kaggle GPU**: Settings → Accelerator → GPU T4 x2 (or P100)
2. **HuggingFace Token**: Add as Kaggle Secret named `HF_TOKEN`
   - Get token: https://huggingface.co/settings/tokens
   - Accept Llama 3.1 license: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
3. **Upload Dataset**: Upload `pubmedQA/splits/train_balanced.json`, `pubmedQA/splits/train_oversampled.json`, and `pubmedQA/splits/dev.json` as a Kaggle Dataset named `pubmedqa-splits`

## Data Split (Official PubMedQA Protocol)
- **Test: 500 samples** (fixed, from official `test_ground_truth.json` — NEVER trained on)
- **Train: 450 samples** (natural distribution: yes=55%, no=34%, maybe=10%)
- **Train Balanced: 141 samples** (47/class — undersampled for unbiased training)
- **Train Oversampled: 747 samples** (249/class — oversampled minority classes)
- **Dev: 50 samples** (for hyperparameter tuning)

**Estimated Time**: ~30-55 min training + 15 min setup = ~1 hour total

## 1. Install Dependencies

In [ ]:
!pip install -q \
    torch \
    transformers>=4.43.0 \
    peft>=0.12.0 \
    bitsandbytes>=0.43.0 \
    trl>=0.9.0 \
    datasets \
    accelerate>=0.33.0 \
    huggingface_hub \
    scikit-learn

print("All dependencies installed!")

## 2. Login & GPU Check

In [ ]:
import torch
import os

# GPU check
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found! Enable GPU in Kaggle: Settings > Accelerator > GPU T4")

# HuggingFace login
from huggingface_hub import login

# On Kaggle, secrets are accessed via:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
except Exception:
    # Fallback for non-Kaggle environments (Colab, local)
    hf_token = os.getenv("HF_TOKEN", "")

if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful!")
else:
    print("WARNING: No HF_TOKEN found. Add it as a Kaggle Secret.")
    print("  1. Go to Add-ons > Secrets")
    print("  2. Add key 'HF_TOKEN' with your token from https://huggingface.co/settings/tokens")

## 3. Load & Prepare PubMedQA Training Data

In [ ]:
import json
from pathlib import Path
from collections import Counter

# ============================================================
# UPDATE THESE PATHS to match your Kaggle Dataset location
# ============================================================
KAGGLE_DATASET_DIR = Path("/kaggle/input/pubmedqa-splits")

# Training data options:
#   train_balanced.json     — 141 samples (47/class, undersampled, RECOMMENDED)
#   train_oversampled.json  — 747 samples (249/class, oversampled minority)
#   train.json              — 450 samples (natural distribution, BIASED — not recommended)
TRAIN_PATH = KAGGLE_DATASET_DIR / "train_oversampled.json"  # Best for fine-tuning
DEV_PATH = KAGGLE_DATASET_DIR / "dev.json"

# Verify files exist
for p in [TRAIN_PATH, DEV_PATH]:
    if not p.exists():
        print(f"ERROR: {p} not found!")
        print("Available files in /kaggle/input/:")
        for f in Path("/kaggle/input").rglob("*.json"):
            print(f"  {f}")
        raise FileNotFoundError(f"Update paths above. {p} not found.")

with open(TRAIN_PATH) as f:
    train_raw = json.load(f)
with open(DEV_PATH) as f:
    dev_raw = json.load(f)

# Verify class balance
train_dist = Counter((v.get("final_decision", "") or "").lower() for v in train_raw.values())
print(f"Train samples: {len(train_raw)}")
print(f"Train distribution: {dict(train_dist)}")
print(f"Dev samples: {len(dev_raw)}")

# Check balance
counts = list(train_dist.values())
if max(counts) - min(counts) > 10:
    print(f"WARNING: Training data is imbalanced! Consider using train_balanced.json")
else:
    print("Training data is balanced.")

In [ ]:
def format_pubmedqa_for_training(data: dict) -> list:
    """
    Convert PubMedQA JSON into instruction-tuning format.
    
    Each sample becomes:
      Input:  Evidence + Question
      Output: Reasoning (LONG_ANSWER) + Final Answer: yes/no/maybe
    
    This teaches the model to:
    1. Read medical evidence
    2. Reason about it
    3. Give a definitive yes/no/maybe answer
    """
    formatted = []
    
    for pubid, item in data.items():
        question = item.get("QUESTION", "").strip()
        contexts = item.get("CONTEXTS", [])
        long_answer = item.get("LONG_ANSWER", "").strip()
        decision = (item.get("final_decision", "") or "").lower().strip()
        
        if not question or decision not in {"yes", "no", "maybe"}:
            continue
        
        # Format evidence
        evidence_parts = []
        for i, ctx in enumerate(contexts):
            evidence_parts.append(f"[Evidence {i+1}] {ctx.strip()}")
        evidence = "\n\n".join(evidence_parts)
        
        # Build instruction prompt (matches our RAG pipeline format)
        instruction = (
            "You are a medical expert answering a yes/no/maybe research question.\n"
            "Based ONLY on the provided evidence, determine if the answer is yes, no, or maybe.\n\n"
            "DECISION RULES:\n"
            "- \"yes\" = evidence SUPPORTS what the question asks\n"
            "- \"no\" = evidence CONTRADICTS what the question asks\n"
            "- \"maybe\" = evidence is truly INCONCLUSIVE\n\n"
            f"EVIDENCE:\n{evidence}\n\n"
            f"QUESTION: {question}\n\n"
            "Provide brief reasoning, then end with your final answer."
        )
        
        # Build response (reasoning + final answer)
        if long_answer:
            response = f"{long_answer}\n\nFinal Answer: {decision}"
        else:
            response = f"Final Answer: {decision}"
        
        formatted.append({
            "instruction": instruction,
            "response": response,
            "pubid": pubid,
            "decision": decision,
        })
    
    return formatted


train_data = format_pubmedqa_for_training(train_raw)
dev_data = format_pubmedqa_for_training(dev_raw)

print(f"Train: {len(train_data)} samples")
print(f"Dev: {len(dev_data)} samples")

# Show class distribution
from collections import Counter
train_dist = Counter(d["decision"] for d in train_data)
print(f"Train distribution: {dict(train_dist)}")

# Show a sample
print("\n" + "="*60)
print("SAMPLE INPUT (first 500 chars):")
print("="*60)
print(train_data[0]["instruction"][:500])
print("\n" + "="*60)
print("SAMPLE OUTPUT:")
print("="*60)
print(train_data[0]["response"][:300])

In [ ]:
from datasets import Dataset

def format_as_chat(sample):
    """
    Format as Llama 3.1 chat template.
    Returns a 'text' field with the full conversation.
    """
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        "You are a medical expert who answers research questions based on evidence. "
        "Always provide brief reasoning and end with 'Final Answer: yes', 'Final Answer: no', "
        "or 'Final Answer: maybe'.<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{sample['instruction']}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{sample['response']}<|eot_id|>"
    )
    return {"text": text}


train_dataset = Dataset.from_list(train_data).map(format_as_chat)
dev_dataset = Dataset.from_list(dev_data).map(format_as_chat)

# Check token lengths
sample_lengths = [len(t.split()) for t in train_dataset["text"]]
print(f"Avg words/sample: {sum(sample_lengths)/len(sample_lengths):.0f}")
print(f"Max words/sample: {max(sample_lengths)}")
print(f"Estimated max tokens: {max(sample_lengths) * 1.3:.0f}")

print(f"\nTrain dataset: {len(train_dataset)} samples")
print(f"Dev dataset: {len(dev_dataset)} samples")
print("\nSample text (first 400 chars):")
print(train_dataset[0]["text"][:400])

## 4. Load Model with QLoRA (4-bit Quantization)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # Nested quantization for extra memory savings
)

print(f"Loading {MODEL_ID} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",  # Memory-efficient attention
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare for training
model = prepare_model_for_kbit_training(model)

print(f"\nModel loaded!")
print(f"Model parameters: {model.num_parameters() / 1e9:.1f}B")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"GPU memory total: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# LoRA Configuration
lora_config = LoraConfig(
    r=16,                          # LoRA rank (16 is a good balance)
    lora_alpha=32,                 # Scaling factor (2x rank is standard)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",      # MLP layers
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Show trainable params
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
print(f"GPU memory after LoRA: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

# Training configuration
training_args = SFTConfig(
    output_dir="./llama31-pubmedqa-lora",
    
    # Training hyperparameters
    num_train_epochs=3,
    per_device_train_batch_size=1,       # Batch 1 to fit in T4
    gradient_accumulation_steps=4,        # Effective batch size = 4
    per_device_eval_batch_size=1,
    
    # Optimizer
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",            # Memory-efficient optimizer
    
    # Memory optimization
    gradient_checkpointing=True,          # Saves ~40% VRAM
    max_seq_length=1024,                  # Our samples fit within this
    fp16=True,
    
    # Logging & eval
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,                   # Keep only 2 best checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Dataset
    dataset_text_field="text",
    packing=False,                        # Don't pack sequences (each is separate)
    
    # Misc
    report_to="none",                     # No W&B etc
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
)

# Print training estimates
total_steps = len(train_dataset) * 3 // 4  # samples * epochs / effective_batch
print(f"Total training steps: ~{total_steps}")
print(f"Estimated time: ~{total_steps * 0.1:.0f} minutes")
print(f"\nStarting training...")

In [ ]:
# TRAIN!
train_result = trainer.train()

# Print results
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Total training time: {train_result.metrics['train_runtime']:.0f} seconds")
print(f"Final train loss: {train_result.metrics['train_loss']:.4f}")
print(f"GPU peak memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## 6. Quick Evaluation (Before Exporting)

In [ ]:
import re

def predict_pubmedqa(model, tokenizer, question, contexts, max_new_tokens=256):
    """Run inference on a single PubMedQA sample."""
    evidence = "\n\n".join(f"[Evidence {i+1}] {c}" for i, c in enumerate(contexts))
    
    messages = [
        {"role": "system", "content": (
            "You are a medical expert who answers research questions based on evidence. "
            "Always provide brief reasoning and end with 'Final Answer: yes', "
            "'Final Answer: no', or 'Final Answer: maybe'."
        )},
        {"role": "user", "content": (
            f"Based ONLY on the following evidence, answer the question.\n\n"
            f"EVIDENCE:\n{evidence}\n\n"
            f"QUESTION: {question}\n\n"
            f"Provide brief reasoning, then end with your final answer."
        )}
    ]
    
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response


def extract_decision(text):
    """Extract yes/no/maybe from model output."""
    t = (text or "").lower().strip()
    m = re.search(r'final\s*answer\s*[:\s]*(yes|no|maybe)', t)
    if m:
        return m.group(1)
    for word in ["yes", "no", "maybe"]:
        if t.startswith(word):
            return word
    return "unknown"


# Evaluate on dev set (99 samples)
print("Evaluating on dev set...")
correct = 0
total = 0
predictions = {"yes": 0, "no": 0, "maybe": 0, "unknown": 0}

for pubid, item in dev_raw.items():
    question = item.get("QUESTION", "")
    contexts = item.get("CONTEXTS", [])
    ground_truth = (item.get("final_decision", "") or "").lower()
    
    if ground_truth not in {"yes", "no", "maybe"}:
        continue
    
    response = predict_pubmedqa(model, tokenizer, question, contexts)
    predicted = extract_decision(response)
    predictions[predicted] = predictions.get(predicted, 0) + 1
    
    if predicted == ground_truth:
        correct += 1
    total += 1
    
    if total <= 3:  # Show first 3
        print(f"\n--- Sample {total} ---")
        print(f"Q: {question[:100]}...")
        print(f"Predicted: {predicted} | Ground Truth: {ground_truth}")
        print(f"Response: {response[:200]}...")
    
    if total % 20 == 0:
        print(f"  Progress: {total}/{len(dev_raw)} ({correct}/{total} correct so far)")

accuracy = correct / total if total > 0 else 0
print(f"\n{'='*60}")
print(f"DEV SET RESULTS (Fine-tuned Llama 3.1 8B)")
print(f"{'='*60}")
print(f"Accuracy: {accuracy:.1%} ({correct}/{total})")
print(f"Predictions: {predictions}")
print(f"{'='*60}")

## 7. Save LoRA Adapter

In [ ]:
import shutil

# Save the LoRA adapter (small: ~50-100MB)
ADAPTER_DIR = "./llama31-pubmedqa-lora/final_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Check size
adapter_size = sum(f.stat().st_size for f in Path(ADAPTER_DIR).rglob("*") if f.is_file())
print(f"Adapter saved to: {ADAPTER_DIR}")
print(f"Adapter size: {adapter_size / 1e6:.1f} MB")
print(f"\nFiles:")
for f in sorted(Path(ADAPTER_DIR).rglob("*")):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Create a zip for easy download
shutil.make_archive(
    "/kaggle/working/llama31-pubmedqa-lora-adapter",
    "zip",
    ADAPTER_DIR
)
zip_size = Path("/kaggle/working/llama31-pubmedqa-lora-adapter.zip").stat().st_size
print(f"\nDownloadable zip: /kaggle/working/llama31-pubmedqa-lora-adapter.zip")
print(f"Zip size: {zip_size / 1e6:.1f} MB")
print("\nDownload this zip from Kaggle Output tab!")

## 8. (Optional) Push to HuggingFace Hub

In [ ]:
# Uncomment to push to your HuggingFace account
# This makes it easy to download later and share

# HF_USERNAME = "your-username"  # <-- Change this
# REPO_NAME = f"{HF_USERNAME}/llama31-8b-pubmedqa-lora"

# model.push_to_hub(REPO_NAME, private=True)
# tokenizer.push_to_hub(REPO_NAME, private=True)
# print(f"Pushed to: https://huggingface.co/{REPO_NAME}")

## 9. How to Use Locally (After Download)

### Option A: With Ollama (Recommended)

```bash
# 1. Install Ollama: https://ollama.ai

# 2. Create a Modelfile
cat > Modelfile <<EOF
FROM llama3.1:8b
ADAPTER ./llama31-pubmedqa-lora-adapter
PARAMETER temperature 0.3
PARAMETER num_predict 512
SYSTEM You are a medical expert who answers research questions based on evidence.
EOF

# 3. Create the custom model
ollama create llama31-medical -f Modelfile

# 4. Test it
ollama run llama31-medical "Does aspirin prevent heart attacks?"

# 5. Use in MEGA-RAG
export OLLAMA_MODEL=llama31-medical
export LLM_PROVIDER=ollama
python run.py --interactive
```

### Option B: Direct Python Loading

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    load_in_4bit=True,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, "./llama31-pubmedqa-lora-adapter")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
```

## 10. Training Summary

In [ ]:
print("="*60)
print("FINE-TUNING SUMMARY")
print("="*60)
print(f"Base Model:       {MODEL_ID}")
print(f"Method:           QLoRA (4-bit NF4 + LoRA r=16)")
print(f"Train Samples:    {len(train_data)}")
print(f"Dev Samples:      {len(dev_data)}")
print(f"Epochs:           3")
print(f"Effective Batch:  4 (1 x 4 gradient accumulation)")
print(f"Max Seq Length:   1024")
print(f"Learning Rate:    2e-4 (cosine schedule)")
print(f"Trainable Params: {trainable:,} ({100 * trainable / total:.2f}%)")
print(f"Train Loss:       {train_result.metrics['train_loss']:.4f}")
print(f"Train Time:       {train_result.metrics['train_runtime']:.0f}s")
print(f"Dev Accuracy:     {accuracy:.1%} ({correct}/{total})")
print(f"Adapter Size:     {adapter_size / 1e6:.1f} MB")
print(f"GPU Peak Memory:  {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print("="*60)

# Save summary as JSON
summary = {
    "base_model": MODEL_ID,
    "method": "QLoRA (4-bit NF4, LoRA r=16, alpha=32)",
    "train_samples": len(train_data),
    "dev_samples": len(dev_data),
    "epochs": 3,
    "train_loss": train_result.metrics['train_loss'],
    "train_runtime_seconds": train_result.metrics['train_runtime'],
    "dev_accuracy": accuracy,
    "dev_predictions": predictions,
    "adapter_size_mb": adapter_size / 1e6,
    "gpu_peak_memory_gb": torch.cuda.max_memory_allocated() / 1e9,
}

with open("/kaggle/working/training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\nSummary saved to /kaggle/working/training_summary.json")